# Notebook 04 - Analyse des Resultats & Rapport Phase 1
**Projet: SentimentsLoupe - Phase 1 Flipkart**

---
## Objectifs
1. Synthese visuelle complete (dashboard recapitulatif)
2. Analyse des erreurs du champion (quels avis sont mal classes?)
3. Courbe d'apprentissage (overfitting?)
4. Rapport structure Phase 1
5. Preparer les conclusions pour la soutenance

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib, json
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import learning_curve, StratifiedKFold

from src.config.settings import (
    MODELS_DIR, PLOTS_DIR, DATA_DIR, FLIPKART_CLEAN, CLASS_NAMES,
    CHAMPION_MODEL_PATH, RANDOM_STATE
)

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'positive': '#1D9E75', 'neutral': '#888780', 'negative': '#E24B4A'}
print('Setup OK')

## 1. Chargement des Resultats

In [ ]:
# Metriques JSON
with open(str(MODELS_DIR / 'phase1_metrics.json'), 'r') as f:
    metrics = json.load(f)

# Resultats experience
results_df = pd.read_csv(str(MODELS_DIR / 'experiment_results.csv'))

# Champion
champion_obj = joblib.load(str(CHAMPION_MODEL_PATH))
champion_model = champion_obj['model']
champion_svd   = champion_obj.get('svd', None)

# Features
data = joblib.load(str(MODELS_DIR / 'features_ready.pkl'))
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

# Dataset nettoye
df_clean = pd.read_csv(str(FLIPKART_CLEAN))

print(f'Champion: [{metrics["best_strategy"]}] {metrics["best_model"]}')
print(f'F1-Macro: {metrics["champion_metrics"]["f1_macro"]:.4f}')

## 2. Dashboard Recapitulatif Phase 1

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle(
    'SentimentsLoupe - Phase 1 Baseline: Tableau de Bord Complet',
    fontsize=16, fontweight='bold', y=0.98
)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# -- 1. KPIs champion (haut gauche) ---------------------------------------
ax_kpi = fig.add_subplot(gs[0, 0])
kpis = [
    ('F1-Macro',   metrics['champion_metrics']['f1_macro'],   '#3B8BD4'),
    ('F1-Negative',metrics['champion_metrics']['f1_negative'],'#E24B4A'),
    ('F1-Neutral', metrics['champion_metrics']['f1_neutral'],  '#888780'),
    ('F1-Positive',metrics['champion_metrics']['f1_positive'], '#1D9E75'),
    ('Accuracy',   metrics['champion_metrics']['accuracy'],    '#7F77DD'),
]
names_kpi, vals_kpi, colors_kpi = zip(*kpis)
bars = ax_kpi.barh(list(names_kpi)[::-1], list(vals_kpi)[::-1],
                   color=list(colors_kpi)[::-1], alpha=0.85, edgecolor='white')
ax_kpi.set_xlim(0, 1.1)
ax_kpi.set_title('Metriques Champion', fontweight='bold')
for bar, v in zip(bars, list(vals_kpi)[::-1]):
    ax_kpi.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

# -- 2. Progression vs dummy ----------------------------------------------
ax_prog = fig.add_subplot(gs[0, 1])
metrics_comp = [
    ('F1-Macro',  metrics['dummy_baseline']['f1_macro'],    metrics['champion_metrics']['f1_macro']),
    ('Accuracy',  metrics['dummy_baseline']['accuracy'],    metrics['champion_metrics']['accuracy']),
]
x = np.arange(len(metrics_comp))
labels_comp  = [m[0] for m in metrics_comp]
dummy_vals   = [m[1] for m in metrics_comp]
champ_vals   = [m[2] for m in metrics_comp]
ax_prog.bar(x - 0.2, dummy_vals, 0.35, label='Sans traitement', color='#D3D1C7', edgecolor='white')
ax_prog.bar(x + 0.2, champ_vals, 0.35, label='Champion', color='#1D9E75', edgecolor='white')
ax_prog.set_title('Champion vs Sans Traitement', fontweight='bold')
ax_prog.set_xticks(x)
ax_prog.set_xticklabels(labels_comp)
ax_prog.legend(fontsize=9)
ax_prog.set_ylim(0, 1.1)
for i, (d, c) in enumerate(zip(dummy_vals, champ_vals)):
    ax_prog.text(i - 0.2, d + 0.02, f'{d:.3f}', ha='center', fontsize=8)
    ax_prog.text(i + 0.2, c + 0.02, f'{c:.3f}', ha='center', fontsize=8, fontweight='bold')

# -- 3. CV scores ---------------------------------------------------------
ax_cv = fig.add_subplot(gs[0, 2])
cv_scores = metrics['cross_validation']['cv_scores']
folds = [f'F{i+1}' for i in range(len(cv_scores))]
ax_cv.bar(folds, cv_scores, color='#3B8BD4', edgecolor='white', alpha=0.85)
mean_cv = metrics['cross_validation']['f1_macro_mean']
std_cv  = metrics['cross_validation']['f1_macro_std']
ax_cv.axhline(y=mean_cv, color='#E24B4A', linestyle='--', linewidth=2)
ax_cv.set_title(f'CV 5-Fold: {mean_cv:.4f} +/- {std_cv:.4f}', fontweight='bold')
ax_cv.set_ylabel('F1-Macro')
for i, v in enumerate(cv_scores):
    ax_cv.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# -- 4. Heatmap F1-Macro tous modeles ------------------------------------
ax_heat = fig.add_subplot(gs[1, :])
heatmap_data = results_df.pivot_table(values='f1_macro', index='strategy', columns='model')
order_strat = ['none', 'class_weight', 'random_over', 'smote', 'smotetomek']
heatmap_data = heatmap_data.reindex([s for s in order_strat if s in heatmap_data.index])
sns.heatmap(
    heatmap_data, ax=ax_heat,
    annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.3, vmax=0.9,
    linewidths=1, linecolor='white',
    cbar_kws={'shrink': 0.6, 'label': 'F1-Macro'}
)
ax_heat.set_title('Heatmap F1-Macro: toutes les combinaisons strategie x modele', fontweight='bold')
ax_heat.set_xlabel('Modele')
ax_heat.set_ylabel('Strategie')

# -- 5. Distribution F1-Negative par strategie ----------------------------
ax_neg = fig.add_subplot(gs[2, 0])
for strategy in results_df['strategy'].unique():
    sub = results_df[results_df['strategy'] == strategy]
    ax_neg.scatter([strategy] * len(sub), sub['f1_negative'],
                   s=60, alpha=0.8, zorder=5)
ax_neg.set_title('F1-Negative par strategie (classe minoritaire)', fontweight='bold')
ax_neg.set_ylabel('F1 Negative')
ax_neg.tick_params(axis='x', rotation=30)
ax_neg.set_ylim(0, 1.0)

# -- 6. Precision vs Recall -----------------------------------------------
ax_pr = fig.add_subplot(gs[2, 1])
for strategy in results_df['strategy'].unique():
    sub = results_df[results_df['strategy'] == strategy]
    ax_pr.scatter(sub['recall_macro'], sub['precision_macro'],
                  label=strategy, s=70, alpha=0.85)
ax_pr.set_title('Precision vs Recall (Macro)', fontweight='bold')
ax_pr.set_xlabel('Recall-Macro')
ax_pr.set_ylabel('Precision-Macro')
ax_pr.legend(fontsize=8)

# -- 7. Top combos --------------------------------------------------------
ax_top = fig.add_subplot(gs[2, 2])
top5 = results_df.head(5)
labels_top = [f"[{r['strategy'][:8]}]\n{r['model'][:12]}" for _, r in top5.iterrows()]
colors_top = ['#FFD700', '#C0C0C0', '#CD7F32', '#3B8BD4', '#3B8BD4']
bars = ax_top.bar(range(len(top5)), top5['f1_macro'],
                  color=colors_top, edgecolor='white', linewidth=1)
ax_top.set_xticks(range(len(top5)))
ax_top.set_xticklabels(labels_top, fontsize=8)
ax_top.set_title('Top 5 Combinaisons (F1-Macro)', fontweight='bold')
ax_top.set_ylabel('F1-Macro')
for bar, v in zip(bars, top5['f1_macro']):
    ax_top.text(bar.get_x() + bar.get_width()/2, v + 0.003,
                f'{v:.4f}', ha='center', fontsize=8, fontweight='bold')

plt.savefig(str(PLOTS_DIR / '04_dashboard_phase1.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Dashboard sauvegarde')

## 3. Analyse des Erreurs du Champion

In [ ]:
# Recuperer les indices test
split_info  = joblib.load(str(MODELS_DIR / 'split_indices.pkl'))
test_indices = split_info['test_indices']
df_test = df_clean.loc[test_indices].copy().reset_index(drop=True)

# Predictions du champion
X_test_use = X_test
if champion_svd is not None:
    from scipy.sparse import issparse
    X_test_use = champion_svd.transform(X_test)

df_test['predicted'] = champion_model.predict(X_test_use)
df_test['predicted_label'] = df_test['predicted'].map({0:'negative', 1:'neutral', 2:'positive'})
df_test['is_correct'] = (df_test['label'] == df_test['predicted']).astype(int)
df_test['true_label'] = df_test['sentiment']

print(f'Taux de bonne classification: {df_test["is_correct"].mean()*100:.2f}%')
print(f'Erreurs totales: {(~df_test["is_correct"].astype(bool)).sum()}')

In [ ]:
# Exemples d'erreurs les plus frequentes
errors = df_test[df_test['is_correct'] == 0]

print('\nExemples d\'erreurs de classification:\n')
for true_class, pred_class in [('negative','positive'), ('negative','neutral'),
                                 ('neutral','positive'), ('positive','negative')]:
    subset = errors[
        (errors['true_label'] == true_class) &
        (errors['predicted_label'] == pred_class)
    ]
    if len(subset) == 0:
        continue
    print(f'--- Reel: {true_class.upper()} | Predi: {pred_class.upper()} ({len(subset)} cas) ---')
    for _, row in subset.head(2).iterrows():
        rev = str(row["review"] if "review" in row else row.get("cleaned_review", ""))
        print(f'  Rating: {row["rating"]} | "{rev[:100]}..."')
    print()

In [ ]:
# Distribution des erreurs par longueur de texte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse des Erreurs du Champion', fontsize=13, fontweight='bold')

# Longueur des avis corrects vs erreurs
if 'word_count_clean' in df_test.columns:
    wc_col = 'word_count_clean'
elif 'word_count' in df_test.columns:
    wc_col = 'word_count'
elif 'review_clean' in df_test.columns:
    df_test['word_count'] = df_test['review_clean'].str.split().str.len()
    wc_col = 'word_count'
else:
    # Fallback: compter les mots dans la review originale
    df_test['word_count'] = df_test['review'].str.split().str.len()
    wc_col = 'word_count'

df_test[df_test['is_correct'] == 1][wc_col].hist(
    bins=30, ax=axes[0], alpha=0.7, color='#1D9E75', label='Correct', density=True)
df_test[df_test['is_correct'] == 0][wc_col].hist(
    bins=30, ax=axes[0], alpha=0.7, color='#E24B4A', label='Erreur', density=True)
axes[0].set_title('Longueur des avis: corrects vs erreurs', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].legend()

# Taux d'erreur par vraie classe
error_rates = df_test.groupby('true_label')['is_correct'].apply(lambda x: (1 - x.mean()) * 100)
bars = axes[1].bar(
    error_rates.index, error_rates.values,
    color=[PALETTE.get(s, '#888') for s in error_rates.index],
    edgecolor='white', linewidth=1
)
axes[1].set_title('Taux d\'erreur par classe reelle', fontweight='bold')
axes[1].set_ylabel('Taux d\'erreur (%)')
for bar, v in zip(bars, error_rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.5,
                 f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '04b_error_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()

## 4. Courbe d'Apprentissage

In [ ]:
print('Calcul de la courbe d\'apprentissage...')

# Preparer les donnees pour la courbe d'apprentissage
if champion_svd is None:
    X_train_lc = X_train
else:
    X_train_lc = champion_svd.transform(X_train)

train_sizes, train_scores, val_scores = learning_curve(
    champion_model,
    X_train_lc, y_train,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 6))
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

ax.plot(train_sizes, train_mean, 'o-', color='#3B8BD4', label='Score entrainement', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#3B8BD4')
ax.plot(train_sizes, val_mean,   's-', color='#E24B4A', label='Score validation CV', linewidth=2)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#E24B4A')

gap_final = train_mean[-1] - val_mean[-1]
ax.set_title(
    f'Courbe d\'Apprentissage - Champion [{metrics["best_strategy"]}] {metrics["best_model"]}\n'
    f'Gap train/val final = {gap_final:.3f}',
    fontweight='bold'
)
ax.set_xlabel('Taille du dataset d\'entrainement')
ax.set_ylabel('F1-Macro')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

if gap_final < 0.05:
    ax.text(0.5, 0.05, 'Pas d\'overfitting detecte', transform=ax.transAxes,
            ha='center', fontsize=11, color='green', fontweight='bold')
else:
    ax.text(0.5, 0.05, f'Overfitting possible (gap={gap_final:.3f})', transform=ax.transAxes,
            ha='center', fontsize=11, color='orange', fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '04c_learning_curve.png'), bbox_inches='tight', dpi=150)
plt.show()

## 5. Rapport Structure Phase 1

In [ ]:
cm = metrics['champion_metrics']
cv = metrics['cross_validation']
dummy = metrics['dummy_baseline']

print('=' * 70)
print('  RAPPORT PHASE 1 - SentimentsLoupe Baseline Flipkart')
print('=' * 70)

print()
print('--- 1. DATASET ---')
print(f'  Source          : Flipkart Product Reviews')
print(f'  Avis totaux     : ~20 000 reviews (dataset Flipkart)')
print(f'  Distribution    : positive=81.1% | negative=10.0% | neutral=8.9%')
print(f'  Desequilibre    : ~8:1 positif/negatif')
print(f'  Split           : 80% train / 20% test (stratifie)')

print()
print('--- 2. PREPROCESSING ---')
print(f'  Nettoyage       : lowercase, URL, emojis, READ MORE, repetitions')
print(f'  Lemmatisation   : WordNetLemmatizer (NLTK)')
print(f'  Features TF-IDF : 15 000 features, n-grammes (1,3), min_df=2')
print(f'  Features num.   : word_count, char_count, avg_word_len, has_excl, caps_ratio')

print()
print('--- 3. STRATEGIE ANTI-DESEQUILIBRE ---')
print(f'  Comparees       : none | class_weight | random_over | SMOTE | SMOTETomek')
print(f'  Classifieurs    : Logistic Regression | LinearSVC | Naive Bayes | SVM-calibre')
print(f'  Total combos    : 18 combinaisons evaluees')

print()
print('--- 4. RESULTATS CHAMPION ---')
print(f'  Dataset         : {metrics["dataset"]}')
print(f'  Strategie       : {metrics["best_strategy"]}')
print(f'  Modele          : {metrics["best_model"]}')
print(f'  F1-Macro        : {cm["f1_macro"]:.4f}  [vs dummy: {dummy["f1_macro"]:.4f} -> +{cm["f1_macro"]-dummy["f1_macro"]:.4f}]')
print(f'  F1-Negative     : {cm["f1_negative"]:.4f}')
print(f'  F1-Neutral      : {cm["f1_neutral"]:.4f}')
print(f'  F1-Positive     : {cm["f1_positive"]:.4f}')
print(f'  Accuracy        : {cm["accuracy"]:.4f}')
print(f'  CV F1-Macro     : {cv["f1_macro_mean"]:.4f} +/- {cv["f1_macro_std"]:.4f}  (5-fold)')

print()
print('--- 5. CONCLUSIONS & SUITE ---')
print(f'  + Pipeline ML valide sur donnees anglaises (Flipkart)')
print(f'  + Desequilibre 8:1 gere efficacement')
print(f'  + Modele stable (CV std = {cv["f1_macro_std"]:.4f})')
print(f'  + Artefacts crees: modeles pkl + metriques JSON + visualisations')
print(f'  + Phase 2: Fine-tuning XLM-RoBERTa sur dataset Darija annote')
print(f'  + Objectif Phase 2: F1-macro > {cm["f1_macro"]:.3f} sur donnees Darija')
print()
print('=' * 70)

In [ ]:
# Liste de tous les fichiers generes Phase 1
import os

print('[Fichiers] generes en Phase 1:')
print()
for root, dirs, files in os.walk(str(DATA_DIR)):
    dirs[:] = [d for d in dirs if d not in ['raw']]
    level = root.replace(str(DATA_DIR), '').count(os.sep)
    indent = '  ' * level
    if level == 0:
        print(f'{os.path.basename(root)}/')
    else:
        print(f'{indent}{os.path.basename(root)}/')
    for file in sorted(files):
        size = os.path.getsize(os.path.join(root, file))
        size_str = f'{size/1024:.1f} KB' if size < 1024*1024 else f'{size/1024/1024:.1f} MB'
        print(f'{indent}  {file:45s} {size_str}')

In [ ]:
print()
print('='*70)
print('  PHASE 1 TERMINEE - Pret pour la Phase 2 (Darija) !')
print('='*70)